要运行此程序，请在 A100 Google Colab Pro 实例上按“*运行时*”并按“*全部运行*”！


<a href="https://github.com/meta-llama/synthetic-data-kit"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/meta%20round%20logo.png" width="137"></a>
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade unsloth {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install synthetic-data-kit==0.0.3
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### 综合数据套件

In [3]:
# 使用 vllm 加载并运行模型
# 我们在前面添加“nohup”并在后面添加“&”以使 Colab 单元在后台运行
! nohup python -m vllm.entrypoints.openai.api_server \
                  --model unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit \
                  --trust-remote-code \
                  --dtype half \
                  --quantization bitsandbytes \
                  --max-model-len 10000 \
                  --tensor-parallel-size 1 \
                  --gpu-memory-utilization 0.7 \
                  --enable-chunked-prefill \
                  --port 8000 \
                  > vllm.log &

nohup: redirecting stderr to stdout


In [4]:
# 尾部 vllm 日志。检查服务器是否已正确启动
!while ! grep -q "Application startup complete" vllm.log; do tail -n 1 vllm.log; sleep 5; done

To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
WARNING 04-29 18:42:44 [config.py:2972] Casting torch.bfloat16 to torch.float16.
WARNING 04-29 18:42:44 [config.py:2972] Casting torch.bfloat16 to torch.float16.
WARNING 04-29 18:42:44 [config.py:2972] Casting torch.bfloat16 to torch.float16.
INFO 04-29 18:43:00 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=2048.
E0000 00:00:1745952186.991164    2914 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1745952186.991164    2914 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 04-29 18

可选：检查 vllm 服务器是否正在运行的函数。将 False 更改为 True 并运行单元格

In [6]:
if False:
  def is_vllm_server_running(api_base_url = None):
      """Simply check if vllm server is running and reachable."""
      print(api_base_url)
      try:
          response = requests.get(f"{api_base_url}/models", timeout = 2)
          return response.status_code == 200
      except:
          return False
  is_running = is_vllm_server_running("http://localhost:8000/v1")
  if is_running:
      print(f"vllm server is running.")
  else:
      print(f"vllm server is not available.")

Text successfully extracted to data/output/ai_meta_com.txt

创建数据目录

In [5]:
!mkdir -p data/{pdf,html,youtube,docx,ppt,txt,output,generated,cleaned,final}

### 摄取源文件

摄取源文件“https://ai.meta.com/blog/llama-4-multimodal-intelligence/”。还可以使用 pdf、docx、ppt 和 youtube 视频

In [6]:
from synthetic_data_kit.core.ingest import process_file
import os

# 直接设置变量
doc_source = "https://ai.meta.com/blog/llama-4-multimodal-intelligence/"
output_dir = "data/output"
name = None  # 让进程自动确定文件名
config = ctx.config if 'ctx' in locals() else None  # 如果可用，请使用 ctx，否则无

try:
    # 直接调用process_file
    output_path = process_file(doc_source, output_dir, name, config)
    print(f"Text successfully extracted to {output_path}")
except Exception as e:
    print(f"Error: {e}")

Text successfully extracted to data/output/ai_meta_com.txt


### 生成 QA 对

在 vllm 和 Llama-3.1-8B-Instruct-unsloth-bnb-4bit 的帮助下生成 QA 对。
将 num_pairs 设置为所需对的数量

In [9]:
from synthetic_data_kit.core.create import process_file
import os
import requests
import json

# 设置参数
input_file = "data/output/ai_meta_com.txt"
output_dir = "data/generated"
config_path = ctx.config_path if 'ctx' in locals() else None  # 如果可用，请使用 ctx
api_base = "http://localhost:8000/v1"  # 默认 vllm API 端点
model = "unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
content_type = "qa"
num_pairs = 10
verbose = False

# 读取输入文件的内容
with open(input_file, 'r') as f:
    text_content = f.read()


print("\n正在生成 QA 对...")
try:
    # 直接使用所有参数调用process_file
    output_path = process_file(
        input_file,
        output_dir,
        config_path,
        api_base,
        model,
        content_type,
        num_pairs,
        verbose
    )

    if output_path:
        print(f"Content saved to {output_path}")

        # 另外，打印生成文件的内容
        try:
            with open(output_path, 'r') as f:
                output_content = f.read()
            print("\n生成的内容（前 500 个字符）：")
            print(output_content[:500] + "..." if len(output_content) > 500 else output_content)
        except Exception as e:
            print(f"Could not read generated file: {e}")
    else:
        print("没有生成输出")
except Exception as e:
    print(f"Error: {e}")

Attempting to terminate the VLLM server

### 整理数据对

In [10]:
from synthetic_data_kit.core.curate import curate_qa_pairs

# 直接设置所有参数
input_file = "data/generated/ai_meta_com_qa_pairs.json"
cleaned_dir = "data/cleaned"
base_name = os.path.splitext(os.path.basename(input_file))[0]
output = os.path.join(cleaned_dir, f"{base_name}_cleaned.json")

threshold = None  # 使用默认阈值
config_path = ctx.config_path if 'ctx' in locals() else None  # 如果可用，请使用 ctx
verbose = False

print("\n正在策划生成的对...")

try:
    # 直接调用 curate_qa_pairs
    result_path = curate_qa_pairs(
        input_file,
        output,
        threshold,
        api_base,
        model,
        config_path,
        verbose
    )

    print(f"Cleaned content saved to {result_path}")

    # 显示清理后的文件内容
    try:
        with open(result_path, 'r') as f:
            output_content = f.read()
        print("\n生成的内容（前 500 个字符）：")
        print(output_content[:500] + "..." if len(output_content) > 500 else output_content)
    except Exception as e:
        print(f"Could not read cleaned file: {e}")
except Exception as e:
    print(f"Error: {e}")

\Curating generated pairs...
Processing 1 batches of QA pairs...
Batch processing complete.
Rated 10 QA pairs
Retained 10 pairs (threshold: 7.0)
Average score: 8.2
Cleaned content saved to data/cleaned/ai_meta_com_qa_pairs_cleaned.json

Generated content (first 500 chars):
{
  "summary": "Here is a summary of the document in 3-5 sentences, focusing on the main topic and key concepts:\n\nMeta AI has announced the Llama 4 herd, a new era of natively multimodal AI innovation, with the release of three models: Llama 4 Scout, Llama 4 Maverick, and Llama 4 Behemoth. These models are designed to enable people to build more personalized multimodal experiences, with Llama 4 Scout and Llama 4 Maverick offering industry-leading performance in image and text understanding, an...


### 保存为 chatML 格式

In [11]:
from synthetic_data_kit.core.save_as import convert_format
import os
import json

# 直接设置所有参数
input_file = "data/cleaned/ai_meta_com_qa_pairs_cleaned.json"
format_type = "ft"  # OpenAI 微调格式
storage_format = "json"  # 默认存储格式

# 设置输出路径
final_dir = "data/final"
# os.makedirs（final_dir，exist_ok = True）
base_name = os.path.splitext(os.path.basename(input_file))[0]

# 确定输出文件路径
if storage_format == "hf":
    output_path = os.path.join(final_dir, f"{base_name}_{format_type}_hf")
else:
    if format_type == "jsonl":
        output_path = os.path.join(final_dir, f"{base_name}.jsonl")
    else:
        output_path = os.path.join(final_dir, f"{base_name}_{format_type}.json")

# 加载配置（如果可用）
config = ctx.config if 'ctx' in locals() else None

try:
    # 直接调用convert_format
    result_path = convert_format(
        input_file,
        output_path,
        format_type,
        config,
        storage_format = storage_format
    )

    print(f"Converted to {format_type} format and saved to {result_path}")

    # 显示转换后文件的内容
    try:
        if os.path.isfile(result_path):
            with open(result_path, 'r') as f:
                output_content = f.read()
            print("\n转换后的内容（前 500 个字符）：")
            print(output_content[:500] + "..." if len(output_content) > 500 else output_content)
        else:
            # 对于 HF 数据集，它是一个目录
            print(f"\nSaved as HF dataset directory at {result_path}")
            if os.path.exists(os.path.join(result_path, "dataset_info.json")):
                with open(os.path.join(result_path, "dataset_info.json"), 'r') as f:
                    info = json.load(f)
                print(f"Dataset info: {info}")
    except Exception as e:
        print(f"Could not read converted file: {e}")

except Exception as e:
    print(f"Error: {e}")

Unsloth 2025.4.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.

In [12]:
# 杀死 vllm 服务器。大约需要 5 秒。
print("尝试终止 vllm 服务器")
!pkill -f "vllm.entrypoints.openai.api_server"

Unsloth 2025.4.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.

### 不懒惰

In [16]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # 选择任何一个！我们内部自动支持 RoPE Scaling！
dtype = None # 没有用于自动检测。 Float16 适用于 Tesla T4、V100，Bfloat16 适用于 Ampere+
load_in_4bit = True # 使用 4 位量化来减少内存使用。可能是假的。

# 我们支持 4 位预量化模型，下载速度提高 4 倍 + 无 OOM。
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # 新的谷歌 6 万亿代币模型速度提高了 2.5 倍！
    "unsloth/gemma-2b-bnb-4bit",
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # 选择任意！例如 teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # 门控模型的 HF 令牌
)

[{'content': 'You are a helpful assistant.', 'role': 'system'},
 {'content': 'What is the context window of Llama 4 Scout?', 'role': 'user'},
 {'content': '10M', 'role': 'assistant'}]

我们现在添加 LoRA 适配器，因此我们只需要更新所有参数的 1 到 10%！

In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 支持任意，但 = 0 已优化
    bias = "none",    # 支持任何，但=“无”已优化
    # [新]“unsloth”使用的 VRAM 减少了 30%，适合 2 倍大的批量大小！
    use_gradient_checkpointing = "unsloth", # 对于很长的上下文来说是真实的或“不懒惰的”
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
)

Unsloth 2025.4.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a名称=“数据”></a>
### 数据准备
我们现在使用“ChatML”格式进行对话风格微调。我们在 ShareGPT 风格中使用 [Open Assistant conversations](https://huggingface.co/datasets/philschmid/guanaco-sharegpt-style)。 ChatML 呈现多轮对话，如下所示：

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What's the capital of France?<|im_end|>
<|im_start|>assistant
Paris.
```

**[注意]** 要仅在完成情况下进行训练（忽略用户的输入），请阅读我们的文档 [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

我们使用“get_chat_template”函数来获取正确的聊天模板。我们支持“zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old”和我们自己优化的“unsloth”模板。

通常我们必须训练`<|im_start|>`和`<|im_end|>`。我们将 `<|im_end|>` 映射为 EOS 代币，并保留 `<|im_start|>` 不变。这不需要额外的令牌的额外训练。

注意 ShareGPT 使用 `{"from": " human", "value" : "Hi"}` 而不是 `{"role": "user", "content" : "Hi"}`，所以我们使用 `mapping` 来映射它。

对于小说写作等文本补全，请尝试此 [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb)。

In [18]:
from unsloth.chat_templates import get_chat_template

# 分词器 = get_chat_template(
# 分词器，
# chat_template = "chatml", # 支持 zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old、unsloth
# 映射 = {"角色" : "来自", "内容" : "值", "用户" : "人类", "助理" : "gpt"}, # ShareGPT 样式
# map_eos_token = True, # 将 <|im_end|> 映射到 </s>
# ）

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset, Dataset
dataset = Dataset.from_json("/content/data/final/ai_meta_com_qa_pairs_cleaned_ft.json")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [19]:
dataset[1]["messages"]

[{'content': 'You are a helpful assistant.', 'role': 'system'},
 {'content': 'What is the context window of Llama 4 Scout?', 'role': 'user'},
 {'content': '10M', 'role': 'assistant'}]

In [20]:
print(dataset[1]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 29 Apr 2025

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the context window of Llama 4 Scout?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

10M<|eot_id|>


如果您想制作自己的聊天模板，这也是可能的！你

---

必须使用 Jinja 模板机制。我们提供了自己的“Unsloth 模板”的精简版本，我们发现它更加高效，并利用了 ChatML、Zephyr 和 Alpaca 风格。

有关 [our wiki page!](https://github.com/unslothai/unsloth/wiki#chat-templates) 上聊天模板的更多信息

In [21]:
unsloth_template = \
    "{{ bos_token }}"\
    "{{ 'You are a helpful assistant to the user\n' }}"\
    "{% for message in messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ '>>> User: ' + message['content'] + '\n' }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ '>>> Assistant: ' + message['content'] + eos_token + '\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '>>> Assistant: ' }}"\
    "{% endif %}"
unsloth_eos_token = "eos_token"

if False:
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = (unsloth_template, unsloth_eos_token,), # 您必须提供模板和 EOS 代币
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # 分享GPT风格
        map_eos_token = True, # 将 <|im_end|> 映射到 </s>
    )

[link text](https://)<a name="Train"></a>
### 训练模型
现在让我们训练我们的模型。我们执行 60 个步骤来加快速度，但您可以设置“num_train_epochs=1”以进行完整运行，并关闭“max_steps=None”。我们还支持`DPOTrainer`和`GRPOTrainer`用于强化学习！

In [22]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # 可以使短序列的训练速度提高 5 倍。
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # 使用 TrackIO/WandB 等
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/10 [00:00<?, ? examples/s]

In [23]:
# @title 显示当前内存统计信息
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
3.441 GB of memory reserved.


In [24]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856/3,000,000,000 (0.81% trained)


Step,Training Loss
1,4.694800
2,4.790200
3,4.632600
4,4.684000
5,4.017900
6,3.816500
7,3.191400
8,2.652200
9,2.399100
10,2.333100


In [25]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

64.247 seconds used for training.
1.07 minutes used for training.
Peak reserved memory = 3.441 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 15.527 %.
Peak reserved memory for training % of max memory = 0.0 %.


*斜体文本*<a name="Inference"></a>
### 推论
让我们运行模型吧！由于我们使用的是“ChatML”，因此请使用“apply_chat_template”并将“add_ Generation_prompt”设置为“True”进行推理。

In [26]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # 支持 zephyr、chatml、mistral、llama、alpaca、vicuna、vicuna_old、unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # 分享GPT风格
    map_eos_token = True, # 将 <|im_end|> 映射到 </s>
)

FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

<|im_start|>user
What is a famous tall tower in Paris?<|im_end|>
<|im_start|>assistant
Eiffel
<|im_end|>assistant

A tower in Paris is called Eiffel.<|im_end|>

您还可以使用 TextStreamer 进行连续推理 - 这样您就可以逐个令牌查看生成令牌，而不是一直等待！

In [27]:
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)

<|im_start|>user
What is a famous tall tower in Paris?<|im_end|>
<|im_start|>assistant
Eiffel
<|im_end|>assistant

A tower in Paris is called Eiffel.<|im_end|>

<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位或 GGUF，请向下滚动！

In [28]:
model.save_pretrained("meta_synthetic_data_lora")  # 本地储蓄
tokenizer.save_pretrained("meta_synthetic_data_lora")
# model.push_to_hub("your_name/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN") # 在线保存

<|im_start|>user
What is a famous tall tower in Paris?<|im_end|>
<|im_start|>assistant
Eiffel
<|im_end|>assistant

A tower in Paris is called Eiffel.<|im_end|>

In [29]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "meta_synthetic_data_lora", # 您用于训练的模型
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

messages = [
    {"from": "human", "value": "What is a famous tall tower in Paris?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # 必须添加生成
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, use_cache = True)

<|im_start|>user
What is a famous tall tower in Paris?<|im_end|>
<|im_start|>assistant
Eiffel
<|im_end|>assistant

A tower in Paris is called Eiffel.<|im_end|>

您还可以使用 Hugging Face 的“AutoPeftModelForCausalLM”。仅当您未安装“unsloth”时才使用此选项。它可能会慢得令人绝望，因为不支持“4bit”模型下载，而且 Unsloth 的**推理速度快了 2 倍**。

In [30]:
if False:
    # 我强烈不建议 - 如果可能的话使用 Unsloth
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        "meta_synthetic_data_lora",  # 您用于训练的模型
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("meta_synthetic_data_lora")

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [31]:
# 合并到16位
if False: model.save_pretrained_merged("meta_synthetic_data_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/meta_synthetic_data_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("meta_synthetic_data_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/meta_synthetic_data_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("meta_synthetic_data_lora")
    tokenizer.save_pretrained("meta_synthetic_data_lora")
if False:
    model.push_to_hub("HF_USERNAME/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/meta_synthetic_data_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

In [32]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer,)
if False: model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("meta_synthetic_data_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/meta_synthetic_data_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  <b>此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可</b>
</div>